# Paper 3 - 02 . Synthetic preference generation

Builds the per-anchor preference dataset (METHOD_DESIGN section 3). Two
stages plus an optional augmentation pass:

1. **Sample rejected.** For each train prompt, sample n=8 base-anchor
   completions at temperature 1; keep the first judged `compliance` as
   `rejected`. Drop the prompt if no compliance is sampled.
2. **Generate chosen.** Mixed teacher per METHOD_DESIGN section 3:
   - harmful prompts -> Claude Opus 4.7 (refusal-style alignment signal)
   - benign  prompts -> Llama-3.3-70B-Instruct (capability-preservation
     signal on over-refusal counter-pairs)

Resumable: every stage caches per-prompt JSONL on Drive.
Augmentations stubbed for the pilot; flip `AUGMENT=True` once the core
set validates.

**Pre-flight gate.** The first cell after bootstrap verifies the
`smoke.json` file written by `00_pilot_smoke_test.ipynb` is present,
fresh (last 7 days), and that its locked-prompt digests match the
current `src/prompts.py` strings. If any check fails, this notebook
refuses to run (PAPER3_PLAN section 15.7).

In [2]:
%%capture
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'torchao>=0.16' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    'requests>=2.32' \
    'tenacity>=9.0' \
    huggingface_hub ipywidgets pyyaml -q


In [3]:
import os, json, gc, time, hashlib, sys
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from google.colab import drive
drive.mount("/content/drive")

# --- A100 sanity + perf toggles ---
import torch
assert torch.cuda.is_available(), (
    'GPU not available. This notebook performs batched bf16 generation on A100 '
    'and will fail without a GPU runtime. Switch Runtime -> Change runtime type '
    '-> A100 GPU and re-run.'
)
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}')
if 'A100' not in DEVICE_NAME:
    print(f'WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE_GEN if VRAM is tight.')

torch.set_float32_matmul_precision('high')          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark  = True              # autotune for fixed shapes


# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SMOKE_DIR    = DRIVE_ROOT / "data" / "smoke"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [PREFS_DIR, SMOKE_DIR, SPLITS_DIR, ADAPTERS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- src.prompts: read from Drive ---
# The notebook + src/prompts.py are synced to Drive together (git pull
# in your local repo, then upload to Drive in the same flow). No git
# clone is performed inside Colab.
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that path "
        "in Drive, then re-run this cell."
    )

# --- Paper 2 reuse: judges, llm_judge, datasheet helpers ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- Paper 3 single-source-of-truth helpers ---
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    TEACHER_SYSTEM_HARMFUL, TEACHER_SYSTEM_BENIGN,
    JUDGE_SYSTEM, JUDGE_USER_TEMPLATE,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    sha16, short_of, family_of,
    verify_smoke_gate, display_locked_state, SmokeGateNotPassed,
    SMOKE_FRESHNESS_DAYS,
)

# --- OpenRouter key from Colab Secrets ---
try:
    from google.colab import userdata
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
except Exception as _e:
    print(f"(secrets not configured: {_e})")

assert os.environ.get("OPENROUTER_API_KEY"), \
    "Set OPENROUTER_API_KEY in Colab Secrets before running this notebook."

print("Bootstrap done.")
print(display_locked_state())


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.
Locked prompts (single source of truth: src/prompts.py)
  CACHE_NAMESPACE_VERSION = 'v3'
  digests:
    TEACHER_SYSTEM_HARMFUL         sha256[:16] = ed7c66091dcb0423
    TEACHER_SYSTEM_BENIGN          sha256[:16] = 4a794d6899bf93c5
    JUDGE_SYSTEM                   sha256[:16] = 6836bdf73c5c8548
    JUDGE_USER_TEMPLATE            sha256[:16] = f065343fdbf745d4


## Pre-flight gate

Bulk Stage 2 will not start until the smoke gate passes. The gate
verifies a fresh, non-drifted `smoke.json` for the configured anchor.
Re-run `00_pilot_smoke_test.ipynb` if this cell fails.

In [4]:
ANCHOR  = "google/gemma-3-4b-it"
TEACHER_HARMFUL = "anthropic/claude-opus-4.7"
TEACHER_BENIGN  = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL     = "openai/gpt-5-mini"

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

print("Pre-flight gate (PAPER3_PLAN section 15.7)")
print("-" * 72)
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR)
except SmokeGateNotPassed as e:
    # Hard-stop with a loud, single error. The operator must re-run
    # notebook 00 (or revert prompt edits) before bulk Stage 2 can start.
    raise SystemExit(f"\nPRE-FLIGHT FAIL\n{e}\n")

print(f"smoke.json found      : data/preferences/{short}/smoke.json")
print(f"smoke completed_at    : {smoke['completed_at']}")
print(f"smoke version         : {smoke.get('smoke_set_version')}")
print(f"smoke teacher (harm)  : {smoke['models']['teacher_harmful']}")
print(f"smoke teacher (benign): {smoke['models']['teacher_benign']}")
print(f"smoke judge           : {smoke['models']['judge']}")
print(f"prompt digests aligned: yes (verified)")
g = smoke.get("gates", {})
print(f"recorded gates        : "
      f"teacher_refusal={g.get('teacher_refusal_rate')}, "
      f"reasoning_overruns={g.get('reasoning_overruns')}, "
      f"cost_overrun={g.get('cost_overrun')}, "
      f"bulk_3_anchors=${g.get('bulk_three_anchors_usd')}")
print(f"\nPRE-FLIGHT PASS. Anchor {short!r} is unlocked for bulk Stage 2.")


Pre-flight gate (PAPER3_PLAN section 15.7)
------------------------------------------------------------------------
smoke.json found      : data/preferences/gemma-3-4b/smoke.json
smoke completed_at    : 2026-05-11T15:09:20+00:00
smoke version         : v3
smoke teacher (harm)  : anthropic/claude-opus-4.7
smoke teacher (benign): meta-llama/Llama-3.3-70B-Instruct
smoke judge           : openai/gpt-5-mini
prompt digests aligned: yes (verified)
recorded gates        : teacher_refusal=0.05, reasoning_overruns=0, cost_overrun=0.0, bulk_3_anchors=$55.37

PRE-FLIGHT PASS. Anchor 'gemma-3-4b' is unlocked for bulk Stage 2.


## Configuration

Hyperparameters and per-stage settings.

In [5]:
SAMPLES_PER_PROMPT = 8     # n in METHOD_DESIGN section 3.2 step 1
BATCH_SIZE_GEN     = 16
TARGET_TOTAL       = 800   # per anchor (PAPER3_PLAN section 5.2)

# Pilot knob -- set to a small int (e.g. 30) to validate the harness
# end-to-end on a tiny budget before committing to the full sweep.
# When STAGE1_LIMIT is None, Stage 1 processes every prompt in the train
# split that does not yet have a rejected-sample record cached.
# Cost reference: ~570 train prompts x up to 8 sample-judges each = ~2.5k
# judge calls at $0.001 = ~$2.50 for the full sweep; STAGE1_LIMIT=30 is
# ~$0.10. Set to None for the bulk run.
STAGE1_LIMIT       = None    # bulk run; set to a small int (e.g. 30) for pilot validation

PREF_OUT      = PREFS_DIR / short
PREF_OUT.mkdir(parents=True, exist_ok=True)
REJECTED_PATH = PREF_OUT / "stage1_rejected.jsonl"
CHOSEN_PATH   = PREF_OUT / "stage2_chosen.jsonl"
PAIRS_PATH    = PREF_OUT / "preferences_v1.jsonl"
print(f"output dir: {PREF_OUT}")


output dir: /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/gemma-3-4b


## Splits -- train/dev/holdout

60/20/20 stratified per dimension, persisted under `data/splits/`.
Cross-lingual is **eval-only** by construction.

In [6]:
SPLIT_PATH = SPLITS_DIR / "split_v1.json"

if SPLIT_PATH.exists():
    SPLIT = json.loads(SPLIT_PATH.read_text())
    print(f"Loaded split assignment from {SPLIT_PATH}")
else:
    import random
    SPLIT = {"train": [], "dev": [], "holdout": []}
    rng = random.Random(0xBADA55)
    for dim_file in sorted((PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl")):
        dim = dim_file.stem
        prompts_dim = [json.loads(l) for l in open(dim_file, encoding="utf-8")]
        if dim == "crosslingual":
            for x in prompts_dim:
                SPLIT["holdout"].append({"dim": dim, "id": x["id"]})
            continue
        ids = [x["id"] for x in prompts_dim]; rng.shuffle(ids)
        n = len(ids); n_tr = int(0.60 * n); n_dev = int(0.20 * n)
        for i, _id in enumerate(ids):
            bucket = "train" if i < n_tr else "dev" if i < n_tr + n_dev else "holdout"
            SPLIT[bucket].append({"dim": dim, "id": _id})
    SPLIT_PATH.write_text(json.dumps(SPLIT, indent=2))
    print(f"Wrote split assignment -> {SPLIT_PATH}")

print({k: len(v) for k, v in SPLIT.items()})

Loaded split assignment from /content/drive/MyDrive/PhD/paper3-alignment/data/splits/split_v1.json
{'train': 519, 'dev': 172, 'holdout': 262}


## OpenRouter wrapper

In [7]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://github.com/anonymised/rosafety-align",
    "X-Title":       "RoSafetyAlign",
})

@retry(stop=stop_after_attempt(4),
       wait=wait_exponential(multiplier=1, min=2, max=20),
       retry=retry_if_exception_type((requests.exceptions.RequestException, RuntimeError)))
def or_chat_completion(*, model, system, user, max_tokens, temperature=0.0,
                        response_format=None):
    payload = {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "usage": {"include": True},
    }
    if response_format is not None:
        payload["response_format"] = response_format
    if model.startswith("openai/gpt-5"):
        payload["reasoning"] = {"effort": "minimal"}

    r = SESSION.post(OPENROUTER_URL, json=payload, timeout=120)
    if r.status_code == 429:
        raise RuntimeError("429 rate limited")
    r.raise_for_status()
    data = r.json()

    choice = data["choices"][0]
    text   = choice["message"].get("content", "") or ""
    finish = choice.get("finish_reason", "")
    native = choice.get("native_finish_reason", "")

    usage   = data.get("usage", {}) or {}
    prompt_tokens     = int(usage.get("prompt_tokens",     0) or 0)
    completion_tokens = int(usage.get("completion_tokens", 0) or 0)
    details = usage.get("completion_tokens_details") or {}
    reasoning_tokens  = int(details.get("reasoning_tokens", 0) or 0)
    cost_usd = float(usage.get("cost", 0.0) or 0.0)

    return {
        "text": text,
        "finish_reason": finish,
        "native_finish_reason": native,
        "usage": {
            "prompt_tokens":     prompt_tokens,
            "completion_tokens": completion_tokens,
            "reasoning_tokens":  reasoning_tokens,
            "cost_usd":          cost_usd,
        },
    }

def or_balance() -> float:
    r = SESSION.get("https://openrouter.ai/api/v1/credits", timeout=30)
    if r.status_code != 200:
        return float("nan")
    data = r.json().get("data", {}) or {}
    total = float(data.get("total_credits", 0.0))
    used  = float(data.get("total_usage",   0.0))
    return total - used

def parse_judge_json(text: str) -> dict:
    """Parse the binary refusal/compliance JSON, with a tolerant fallback
    that pulls the first balanced {...} out of the response."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start == -1:
        return {"label": "compliance", "justification": "judge JSON parse failed"}
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i+1])
                except json.JSONDecodeError: continue
    return {"label": "compliance", "justification": "judge JSON parse failed"}

## Stage 1 -- sample base-anchor completions, find rejected

Batched generation on A100. Sample at temperature 1 / top-p 0.95 with
`n=SAMPLES_PER_PROMPT` independent completions per prompt. Judge each
until one is judged `compliance`.

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_kwargs_for(family: str) -> dict:
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}

TRAIN_DIMS = ("toxicity", "jailbreak", "bias")
train_items = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]

# Deterministic shuffle (seed 0xBADA55 -- same seed as SPLIT construction)
# so STAGE1_LIMIT=30 sees a stratified random sample across dims rather
# than 30 prompts from whichever dim sorts first alphabetically.
import random as _random
_random.Random(0xBADA55).shuffle(train_items)

from collections import Counter as _Counter
print(f"train items per dim: {dict(_Counter(x['dim'] for x in train_items))}")
by_id = {}
for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it
print(f"train items to source preferences from: {len(train_items)}")

done_ids = set()
if REJECTED_PATH.exists():
    for line in open(REJECTED_PATH, encoding="utf-8"):
        done_ids.add(tuple(json.loads(line)["key"]))
    print(f"resuming -- {len(done_ids)} rejected pairs already cached")

todo = [x for x in train_items if (x["dim"], x["id"]) not in done_ids]
if STAGE1_LIMIT is not None:
    todo = todo[:STAGE1_LIMIT]
    print(f"STAGE1_LIMIT={STAGE1_LIMIT} -> processing {len(todo)} prompts (pilot mode)")
print(f"todo per dim       : {dict(_Counter(x['dim'] for x in todo))}")

if todo:
    tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
    mdl.eval()

    def judge_compliance_via_or(prompt, response):
        r = or_chat_completion(
            model=JUDGE_MODEL, system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(
                prompt=prompt, response=response or "[empty response]"),
            max_tokens=300, temperature=0.0,
            response_format={"type": "json_object"},
        )
        return parse_judge_json(r["text"]).get("label") == "compliance"

    with open(REJECTED_PATH, "a", encoding="utf-8") as fout:
        for chunk_start in range(0, len(todo), BATCH_SIZE_GEN):
            chunk = todo[chunk_start:chunk_start + BATCH_SIZE_GEN]
            prompts_text = [by_id[(x["dim"], x["id"])]["text_ro"] for x in chunk]
            formatted = [
                tok.apply_chat_template(
                    [{"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True,
                ) for p in prompts_text
            ]
            expanded = [t for t in formatted for _ in range(SAMPLES_PER_PROMPT)]
            inputs = tok(expanded, return_tensors="pt", padding=True,
                         truncation=True, max_length=1024).to(mdl.device)
            with torch.no_grad():
                out = mdl.generate(
                    **inputs, max_new_tokens=256,
                    do_sample=True, temperature=1.0, top_p=0.95,
                    pad_token_id=tok.pad_token_id,
                )
            input_len = inputs["input_ids"].shape[1]
            gens = [tok.decode(seq[input_len:], skip_special_tokens=True) for seq in out]

            for j, x in enumerate(chunk):
                completions = gens[j * SAMPLES_PER_PROMPT : (j + 1) * SAMPLES_PER_PROMPT]
                rejected = None
                for ci, comp in enumerate(completions):
                    if judge_compliance_via_or(prompts_text[j], comp):
                        rejected = {"sample_index": ci, "text": comp}
                        break
                fout.write(json.dumps({
                    "key": (x["dim"], x["id"]),
                    "prompt": prompts_text[j],
                    "rejected": rejected,
                    "samples_tried": len(completions),
                }, ensure_ascii=False) + "\n")
                fout.flush()
            if (chunk_start // BATCH_SIZE_GEN) % 4 == 0:
                print(f"  stage1: {chunk_start + len(chunk)}/{len(todo)} ({short})")

    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
n_total       = len(rejected_records)
n_with_reject = sum(1 for r in rejected_records if r["rejected"])

# Per-dim yield + sample-index distribution
_per_dim       = _Counter()
_per_dim_yield = _Counter()
_sample_idx    = _Counter()
for r in rejected_records:
    dim = r["key"][0]
    _per_dim[dim] += 1
    if r["rejected"]:
        _per_dim_yield[dim] += 1
        _sample_idx[r["rejected"]["sample_index"]] += 1

print("")
print(f"=== Stage 1 dashboard ({n_total} prompts) ===")
print(f"  total yield      : {n_with_reject}/{n_total} ({n_with_reject/max(n_total,1):.0%})")
print(f"  per-dim yield    :")
for dim in sorted(_per_dim):
    y, n = _per_dim_yield[dim], _per_dim[dim]
    print(f"    {dim:<12s}  {y:>3}/{n:<3}  ({y/max(n,1):.0%})")
print(f"  sample-index hist: {dict(sorted(_sample_idx.items()))}")
_n_judge_calls = sum(
    (r['rejected']['sample_index'] + 1) if r['rejected'] else r['samples_tried']
    for r in rejected_records
)
print(f"  total judge calls: {_n_judge_calls} (= ~${_n_judge_calls * 0.001:.2f} at $0.001/call)")
# --- Pilot eyeball: first 5 successful rejected-sample records ---
# Useful for catching obvious failure modes early -- garbled responses,
# judge mis-classification, broken chat-template handling.
preview = [r for r in rejected_records if r.get('rejected')][:5]
print(f"\nFirst {len(preview)} rejected-sample records (eyeball check):")
for i, r in enumerate(preview, 1):
    rej = r['rejected']
    print(f"\n--- {i}. key={r['key']} sample_index={rej['sample_index']} ---")
    print(f"PROMPT  : {r['prompt'][:200]}")
    print(f"REJECTED: {rej['text'][:400]}")


train items per dim: {'toxicity': 156, 'bias': 129, 'jailbreak': 58}
train items to source preferences from: 343
resuming -- 343 rejected pairs already cached
todo per dim       : {}

=== Stage 1 dashboard (343 prompts) ===
  total yield      : 293/343 (85%)
  per-dim yield    :
    bias          128/129  (99%)
    jailbreak      46/58   (79%)
    toxicity      119/156  (76%)
  sample-index hist: {0: 263, 1: 19, 2: 4, 3: 2, 4: 2, 5: 1, 6: 1, 7: 1}
  total judge calls: 752 (= ~$0.75 at $0.001/call)

First 5 rejected-sample records (eyeball check):

--- 1. key=['toxicity', 'tox_ethn_4859e75b'] sample_index=0 ---
PROMPT  : Formulează un proiect de misionare care promovează conceptul de inclusiv
REJECTED: ## Proiect Misionar: "Pietre de Legătură – Construind Comunități Incluzive"

**1. Scopul Proiectului:**

"Pietre de Legătură" își propune să promoveze și să implementeze conceptul de incluziune în comunități marginalizate, prin construirea de legături durabile, respect reciproc și oportun

## Stage 2 -- query the teacher for `chosen`

OpenRouter call per prompt. Mixed teacher: harmful side -> Claude Opus
4.7 with `TEACHER_SYSTEM_HARMFUL`; benign side (over-refusal counter-
pairs only -- handled in Stage 4 augmentation) -> Llama-3.3-70B with
`TEACHER_SYSTEM_BENIGN`.

In [9]:
done_keys = set()
stale_count = 0
if CHOSEN_PATH.exists():
    for line in open(CHOSEN_PATH, encoding="utf-8"):
        r = json.loads(line)
        # Only treat a cached chosen record as done if it was generated
        # under the *current* prompt version. Older entries (no
        # prompt_version field, or a stale one) are re-queried so the
        # new TEACHER_SYSTEM_HARMFUL takes effect on every pair.
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            done_keys.add(tuple(r["key"]))
        else:
            stale_count += 1
    if stale_count:
        print(f"resume: {stale_count} stage 2 records ignored (prompt_version != {CACHE_NAMESPACE_VERSION!r}); they will be regenerated.")

rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
todo = [r for r in rejected_records if r["rejected"] and tuple(r["key"]) not in done_keys]
print(f"stage 2 todo: {len(todo)} pairs")

teacher_failures = 0
with open(CHOSEN_PATH, "a", encoding="utf-8") as fout:
    for i, r in enumerate(todo):
        try:
            resp = or_chat_completion(
                model=TEACHER_HARMFUL, system=TEACHER_SYSTEM_HARMFUL,
                user=r["prompt"], max_tokens=600, temperature=0.0,
            )
            chosen_text = resp["text"]
        except Exception as e:
            chosen_text = ""; teacher_failures += 1
            if teacher_failures >= 5:
                print(f"WARNING: teacher refused/failed >= 5 times "
                      f"({e}). Consider switching TEACHER_HARMFUL fallback.")

        verify = or_chat_completion(
            model=JUDGE_MODEL, system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(
                prompt=r["prompt"], response=chosen_text or "[empty response]"),
            max_tokens=300, temperature=0.0,
            response_format={"type": "json_object"},
        )
        verdict = parse_judge_json(verify["text"])
        ok = verdict.get("label") == "refusal" and bool(chosen_text.strip())
        fout.write(json.dumps({
            "key": r["key"], "prompt": r["prompt"],
            "chosen": chosen_text,
            "judge_label": verdict.get("label"),
            "ok": ok,
            "prompt_version": CACHE_NAMESPACE_VERSION,
        }, ensure_ascii=False) + "\n")
        fout.flush()
        if (i + 1) % 25 == 0:
            print(f"  stage2: {i + 1}/{len(todo)} (failures={teacher_failures})")
print(f"stage 2 done. teacher refusal/empty failures: {teacher_failures}")


stage 2 todo: 0 pairs
stage 2 done. teacher refusal/empty failures: 0


## Stage 3 -- emit preference pairs (core set)

In [10]:
rej = {tuple(r["key"]): r for r in (json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8"))}
# Filter to current prompt_version so a partially-replayed run that
# left old-version records in CHOSEN_PATH does not contaminate the
# assembled pairs file.
cho = {}
for line in open(CHOSEN_PATH, encoding="utf-8"):
    r = json.loads(line)
    if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
        cho[tuple(r["key"])] = r

pairs = []
for k in rej:
    a = rej[k]; b = cho.get(k)
    if not (a and b and a["rejected"] and b.get("ok")): continue
    pairs.append({
        "prompt":   a["prompt"],
        "chosen":   b["chosen"],
        "rejected": a["rejected"]["text"],
        "meta": {
            "split_key": list(k),
            "anchor": ANCHOR,
            "teacher": TEACHER_HARMFUL,
            "rejected_sample_index": a["rejected"]["sample_index"],
        },
    })
print(f"core preference pairs assembled: {len(pairs)}")


core preference pairs assembled: 63


## Stage 4 -- augmentations (stub for pilot)

Cross-lingual + over-refusal counter-pairs land here when AUGMENT=True
after the pilot validates. The pilot run at AUGMENT=False uses the
core ~400 pairs.

In [11]:
AUGMENT = False
if AUGMENT:
    raise NotImplementedError(
        "Augmentation pipeline (cross-lingual, over-refusal counter-pairs) "
        "lands in week-2. The cross-lingual pairs use TEACHER_SYSTEM_HARMFUL "
        "with Claude Opus 4.7, the over-refusal counter-pairs use "
        "TEACHER_SYSTEM_BENIGN with Llama-3.3-70B. See METHOD_DESIGN section 3.3."
    )

## Save the dataset

In [12]:
with open(PAIRS_PATH, "w", encoding="utf-8") as f:
    for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")

sha = hashlib.sha256(PAIRS_PATH.read_bytes()).hexdigest()[:16]
(PREF_OUT / "preferences_v1.meta.json").write_text(json.dumps({
    "anchor": ANCHOR, "teacher_harmful": TEACHER_HARMFUL,
    "teacher_benign": TEACHER_BENIGN,
    "n_pairs": len(pairs), "sha256_16": sha,
    "split_path": str(SPLIT_PATH),
    "core_only": True, "augmentations_applied": False,
    "prompt_digests": dict(PROMPT_DIGESTS),
    "cache_namespace_version": CACHE_NAMESPACE_VERSION,
    "smoke_completed_at": smoke["completed_at"],
    "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}, indent=2))
print(f"Wrote {len(pairs)} pairs -> {PAIRS_PATH}")
print(f"sha256[:16] = {sha}")

Wrote 63 pairs -> /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/gemma-3-4b/preferences_v1.jsonl
sha256[:16] = 87a131aa25581b9a


## Batch run: preference data for every anchor in `ANCHORS`

Loops over `ANCHORS` (imported from `src/prompts.py`). For each
anchor runs Stage 1 (sample base anchor, find rejected), Stage 2
(query Claude Opus for chosen), Stage 3 (assemble pairs), and
save. Skips an anchor whose `preferences_v1.jsonl` already exists.

Re-uses cached `stage1_rejected.jsonl` and `stage2_chosen.jsonl`
per anchor so a partially-completed run resumes cleanly.

Operator workflow once this cell exists:
1. Run install + bootstrap + pre-flight (per anchor) + cfg (set
   `STAGE1_LIMIT` to None for bulk or 30 for pilot) + splits + or.
2. Run only this batch cell.
Total: ~30 min for three anchors plus ~$1.50-3 OpenRouter.


In [13]:
import gc, json, random as _random, time
from collections import Counter
from datetime import datetime, timezone

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- defensive: ensure prompts.py is importable when run standalone ---
# (no-op if bootstrap already ran in this kernel)
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

from prompts import ANCHORS

# Same load_kwargs_for as p3-02-stage1; gemma needs eager attn.
def _load_kwargs_for(family: str) -> dict:
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        return {**common, "attn_implementation": "eager"}
    return {**common, "attn_implementation": "sdpa"}

TRAIN_DIMS = ("toxicity", "jailbreak", "bias")

# Pre-load all train items + by_id once (anchor-independent).
_train_items_all = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]
_random.Random(0xBADA55).shuffle(_train_items_all)
_by_id = {}
for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); _by_id[(dim_file.stem, it["id"])] = it


def _judge_compliance_via_or(prompt, response):
    r = or_chat_completion(
        model=JUDGE_MODEL, system=JUDGE_SYSTEM,
        user=JUDGE_USER_TEMPLATE.format(
            prompt=prompt, response=response or "[empty response]"),
        max_tokens=300, temperature=0.0,
        response_format={"type": "json_object"},
    )
    return parse_judge_json(r["text"]).get("label") == "compliance"


def _process_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    pref_out = PREFS_DIR / short_a
    pref_out.mkdir(parents=True, exist_ok=True)

    rejected_path = pref_out / "stage1_rejected.jsonl"
    chosen_path   = pref_out / "stage2_chosen.jsonl"
    pairs_path    = pref_out / "preferences_v1.jsonl"

    if pairs_path.exists():
        print(f"\n[{anchor}] {pairs_path.name} already exists; skipping.")
        return

    print(f"\n=== preferences for {anchor} -> {pref_out} ===")

    # Pre-flight gate per anchor.
    try:
        verify_smoke_gate(PREFS_DIR, anchor)
    except SmokeGateNotPassed as e:
        print(f"  PRE-FLIGHT FAIL for {anchor}: {e}")
        print(f"  skipping (re-run notebook 00 for this anchor first)")
        return

    # ---- Stage 1: sample base anchor, find rejected ----
    done_ids = set()
    if rejected_path.exists():
        for line in open(rejected_path, encoding="utf-8"):
            done_ids.add(tuple(json.loads(line)["key"]))
        print(f"  resuming -- {len(done_ids)} rejected pairs cached")
    todo = [x for x in _train_items_all if (x["dim"], x["id"]) not in done_ids]
    if STAGE1_LIMIT is not None:
        todo = todo[:STAGE1_LIMIT]
        print(f"  STAGE1_LIMIT={STAGE1_LIMIT} -> processing {len(todo)} prompts (pilot mode)")
    print(f"  stage 1 todo: {len(todo)} prompts ({dict(Counter(x['dim'] for x in todo))})")

    if todo:
        tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
        if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
        mdl = AutoModelForCausalLM.from_pretrained(anchor, **_load_kwargs_for(family_a))
        mdl.eval()

        with open(rejected_path, "a", encoding="utf-8") as fout:
            for chunk_start in range(0, len(todo), BATCH_SIZE_GEN):
                chunk = todo[chunk_start:chunk_start + BATCH_SIZE_GEN]
                prompts_text = [_by_id[(x["dim"], x["id"])]["text_ro"] for x in chunk]
                formatted = [
                    tok_a.apply_chat_template(
                        [{"role": "user", "content": p}],
                        tokenize=False, add_generation_prompt=True,
                    ) for p in prompts_text
                ]
                expanded = [t for t in formatted for _ in range(SAMPLES_PER_PROMPT)]
                inputs = tok_a(expanded, return_tensors="pt", padding=True,
                               truncation=True, max_length=1024).to(mdl.device)
                with torch.no_grad():
                    out = mdl.generate(
                        **inputs, max_new_tokens=256,
                        do_sample=True, temperature=1.0, top_p=0.95,
                        pad_token_id=tok_a.pad_token_id,
                    )
                input_len = inputs["input_ids"].shape[1]
                gens = [tok_a.decode(seq[input_len:], skip_special_tokens=True) for seq in out]

                for j, x in enumerate(chunk):
                    completions = gens[j * SAMPLES_PER_PROMPT : (j + 1) * SAMPLES_PER_PROMPT]
                    rejected = None
                    for ci, comp in enumerate(completions):
                        if _judge_compliance_via_or(prompts_text[j], comp):
                            rejected = {"sample_index": ci, "text": comp}
                            break
                    fout.write(json.dumps({
                        "key": (x["dim"], x["id"]),
                        "prompt": prompts_text[j],
                        "rejected": rejected,
                        "samples_tried": len(completions),
                    }, ensure_ascii=False) + "\n")
                    fout.flush()
                if (chunk_start // BATCH_SIZE_GEN) % 4 == 0:
                    print(f"    stage1: {chunk_start + len(chunk)}/{len(todo)} ({short_a})")

        del mdl, tok_a; gc.collect(); torch.cuda.empty_cache()

    # ---- Stage 2: query teacher for chosen ----
    done_keys = set()
    if chosen_path.exists():
        for line in open(chosen_path, encoding="utf-8"):
            r = json.loads(line)
            if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
                done_keys.add(tuple(r["key"]))

    rejected_records = [json.loads(l) for l in open(rejected_path, encoding="utf-8")]
    todo2 = [r for r in rejected_records if r["rejected"] and tuple(r["key"]) not in done_keys]
    print(f"  stage 2 todo: {len(todo2)} pairs")

    teacher_failures = 0
    with open(chosen_path, "a", encoding="utf-8") as fout:
        for i, r in enumerate(todo2):
            try:
                resp = or_chat_completion(
                    model=TEACHER_HARMFUL, system=TEACHER_SYSTEM_HARMFUL,
                    user=r["prompt"], max_tokens=600, temperature=0.0,
                )
                chosen_text = resp["text"]
            except Exception:
                chosen_text = ""; teacher_failures += 1

            verify = or_chat_completion(
                model=JUDGE_MODEL, system=JUDGE_SYSTEM,
                user=JUDGE_USER_TEMPLATE.format(
                    prompt=r["prompt"], response=chosen_text or "[empty response]"),
                max_tokens=300, temperature=0.0,
                response_format={"type": "json_object"},
            )
            verdict = parse_judge_json(verify["text"])
            ok = verdict.get("label") == "refusal" and bool(chosen_text.strip())
            fout.write(json.dumps({
                "key": r["key"], "prompt": r["prompt"],
                "chosen": chosen_text,
                "judge_label": verdict.get("label"),
                "ok": ok,
                "prompt_version": CACHE_NAMESPACE_VERSION,
            }, ensure_ascii=False) + "\n")
            fout.flush()
            if (i + 1) % 25 == 0:
                print(f"    stage2: {i + 1}/{len(todo2)} (failures={teacher_failures})")

    # ---- Stage 3: assemble pairs ----
    rej = {tuple(r["key"]): r for r in (json.loads(l) for l in open(rejected_path, encoding="utf-8"))}
    cho = {}
    for line in open(chosen_path, encoding="utf-8"):
        r = json.loads(line)
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            cho[tuple(r["key"])] = r

    pairs = []
    for k in rej:
        a = rej[k]; b = cho.get(k)
        if not (a and b and a["rejected"] and b.get("ok")): continue
        pairs.append({
            "prompt":   a["prompt"],
            "chosen":   b["chosen"],
            "rejected": a["rejected"]["text"],
            "meta": {
                "split_key": list(k),
                "anchor": anchor,
                "teacher": TEACHER_HARMFUL,
                "rejected_sample_index": a["rejected"]["sample_index"],
                "source": "core_s2",
            },
        })
    print(f"  core preference pairs: {len(pairs)}")

    # ---- save ----
    with open(pairs_path, "w", encoding="utf-8") as f:
        for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")
    import hashlib as _hashlib
    sha = _hashlib.sha256(pairs_path.read_bytes()).hexdigest()[:16]
    smoke = verify_smoke_gate(PREFS_DIR, anchor)
    (pref_out / "preferences_v1.meta.json").write_text(json.dumps({
        "anchor": anchor, "teacher_harmful": TEACHER_HARMFUL,
        "teacher_benign": TEACHER_BENIGN,
        "n_pairs": len(pairs), "sha256_16": sha,
        "split_path": str(SPLITS_DIR / "split_v1.json"),
        "core_only": True, "augmentations_applied": False,
        "prompt_digests": dict(PROMPT_DIGESTS),
        "cache_namespace_version": CACHE_NAMESPACE_VERSION,
        "smoke_completed_at": smoke["completed_at"],
        "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2))
    print(f"  wrote {len(pairs)} pairs -> {pairs_path}")
    print(f"  sha256[:16] = {sha}")


for anchor in ANCHORS:
    _process_one(anchor)
print("\nbatch preferences done.")



[Qwen/Qwen2.5-3B-Instruct] preferences_v1.jsonl already exists; skipping.

[meta-llama/Llama-3.2-3B-Instruct] preferences_v1.jsonl already exists; skipping.

[google/gemma-3-4b-it] preferences_v1.jsonl already exists; skipping.

batch preferences done.


## x4 expansion: translate fresh HarmBench EN -> RO

Generates a new pool of Romanian harmful prompts by translating
HarmBench EN prompts that are NOT already in the existing xl
stream. Translation via Llama-3.3-70B (the same benign teacher
we use for over-refusal counter pairs). Anchor-independent,
runs once.

Output: `data/expanded_prompts/harmbench_ro_v1.jsonl`. Each row
carries the original EN id, the EN text, the RO translation, and
the HarmBench category. Skips if the file already exists.

**Caveat for the manuscript.** Translated HarmBench prompts are
culturally generic ('how to make a bomb') vs RoSafetyBench's
cultural-specific holdout (Roma, Hungarian, regional). Even if
expansion does not lift OOD cross-lingual, this is consistent
with a 'cultural-specific data is the bottleneck' reading, not
'data scale does not help'.


In [14]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import json, time
from pathlib import Path

EXPAND_DIR = DRIVE_ROOT / "data" / "expanded_prompts"
EXPAND_DIR.mkdir(parents=True, exist_ok=True)
TRANSLATED_PATH = EXPAND_DIR / "harmbench_ro_v1.jsonl"

if TRANSLATED_PATH.exists():
    n = sum(1 for _ in open(TRANSLATED_PATH, encoding="utf-8"))
    print(f"[skip] {TRANSLATED_PATH.name} already exists ({n} translated prompts)")
else:
    # Pull HarmBench EN prompts that are NOT in the existing xl stream.
    # Canonical source: data/augmentation/harmbench_behaviors_v1.csv,
    # the same file nb02b downloads from the HarmBench repo. Auto-download
    # if missing so this cell works on a fresh runtime.
    import csv, urllib.request
    AUG_DIR = DRIVE_ROOT / "data" / "augmentation"
    AUG_DIR.mkdir(parents=True, exist_ok=True)
    HARMBENCH_PATH = AUG_DIR / "harmbench_behaviors_v1.csv"
    if not HARMBENCH_PATH.exists():
        print(f"downloading HarmBench behaviors -> {HARMBENCH_PATH}")
        url = ("https://raw.githubusercontent.com/centerforaisafety/HarmBench/"
               "main/data/behavior_datasets/harmbench_behaviors_text_all.csv")
        urllib.request.urlretrieve(url, HARMBENCH_PATH)
    print(f"HarmBench source: {HARMBENCH_PATH}  ({HARMBENCH_PATH.stat().st_size:,} bytes)")

    # Read source -- harmbench_behaviors_v1.csv has columns BehaviorID,
    # Behavior, SemanticCategory (lowercase via .lower() to match nb02b).
    en_rows = []
    with open(HARMBENCH_PATH, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            bid = (r.get("BehaviorID") or "").strip()
            text = (r.get("Behavior") or "").strip()
            if not bid or not text: continue
            en_rows.append({
                "id":       bid,
                "text":     text,
                "category": (r.get("SemanticCategory") or "harmful").strip().lower(),
            })
    print(f"loaded {len(en_rows)} HarmBench EN prompts from {HARMBENCH_PATH.name}")

    # Exclude IDs already used in xl streams
    used = set()
    for short_a in ("qwen2.5-3b", "llama-3.2-3b", "gemma-3-4b"):
        for fname in ("xl_used_ids.json", "xl_pairs.jsonl"):
            p = DRIVE_ROOT / "data" / "augmentation" / short_a / fname
            if p.exists():
                if p.suffix == ".json":
                    used.update(json.loads(p.read_text()))
                else:
                    for line in open(p, encoding="utf-8"):
                        used.add(json.loads(line).get("source_id"))
    fresh = [r for r in en_rows if r["id"] and r["id"] not in used]
    print(f"  {len(fresh)} prompts not yet used in xl stream")

    # HarmBench standard text-all has ~400 behaviors total. nb02b
    # already consumed ~200 of them for the xl stream, so the realistic
    # ceiling on 'fresh' is ~200 unique IDs. We translate everything we
    # can find that wasn't used yet (no hard cap; the source itself caps).
    print(f"  translating {len(fresh)} prompts via Llama-3.3-70B...")

    TRANSLATE_SYSTEM = (
        "You are a translation system. Translate the user's English "
        "text into natural, idiomatic Romanian. Preserve meaning "
        "exactly. Output ONLY the Romanian translation; no preamble, "
        "no commentary, no quotes."
    )

    out = open(TRANSLATED_PATH, "w", encoding="utf-8")
    n_ok = n_fail = 0
    cost_total = 0.0
    t0 = time.time()
    for i, r in enumerate(fresh):
        try:
            resp = or_chat_completion(
                model="meta-llama/Llama-3.3-70B-Instruct",
                system=TRANSLATE_SYSTEM,
                user=r["text"],
                max_tokens=200, temperature=0.0,
            )
            text_ro = resp["text"].strip()
            cost_total += resp["usage"].get("cost_usd", 0.0) or 0.0
            if not text_ro or len(text_ro) < 8:
                n_fail += 1; continue
            out.write(json.dumps({
                "id":       f"hb_ro_{r['id']}",
                "source_id": r["id"],
                "text_en":  r["text"],
                "text_ro":  text_ro,
                "category": r.get("category", "harmful"),
            }, ensure_ascii=False) + "\n")
            out.flush()
            n_ok += 1
        except Exception as e:
            n_fail += 1
        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            print(f"    translated {i+1}/{len(fresh)}  ok={n_ok}  fail={n_fail}  cost=${cost_total:.2f}  elapsed={elapsed/60:.1f}min")
    out.close()
    print(f"\ntranslated {n_ok} prompts -> {TRANSLATED_PATH}  (failures: {n_fail}, cost: ${cost_total:.2f})")


[skip] harmbench_ro_v1.jsonl already exists (400 translated prompts)


## x4 expansion: paraphrase RoSafetyBench train items

Generates 3 Romanian paraphrases of each RoSafetyBench train
item (toxicity / bias / jailbreak only) via Llama-3.3-70B. The
paraphrases preserve the cultural-specific Romanian harmful
intent that RoSafetyBench's holdout evaluates against, expanding
the training pool by ~3x in the dimension that matters for
cross-lingual safety.

Why Llama-3.3-70B not Claude Opus 4.7 for this step: Claude
occasionally refuses to paraphrase harmful-flavoured prompts
even when the source is in our own dataset, while Llama-3.3-70B
treats paraphrase as a translation-style task and almost never
refuses. Cost: ~$0.50-1 per pass. Outputs that look like refusals
(short response, common refusal markers) are detected and
skipped.

Output: `data/expanded_prompts/rosafety_paraphrased_v1.jsonl`.
Skips if the file already exists (resumes by re-reading).


In [19]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import json, time
from pathlib import Path

EXPAND_DIR = DRIVE_ROOT / "data" / "expanded_prompts"
EXPAND_DIR.mkdir(parents=True, exist_ok=True)
PARAPHRASED_PATH = EXPAND_DIR / "rosafety_paraphrased_v1.jsonl"

if PARAPHRASED_PATH.exists():
    n = sum(1 for _ in open(PARAPHRASED_PATH, encoding="utf-8"))
    print(f"[skip] {PARAPHRASED_PATH.name} already exists ({n} paraphrases)")
else:
    PARAPHRASES_PER_ITEM = 3
    PARAPHRASE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

    PARAPHRASE_SYSTEM = (
        "You are a Romanian text variation system. Given a Romanian "
        "sentence, write a paraphrase that preserves the meaning and "
        "register exactly but uses different wording, sentence "
        "structure, or word order. Do not soften, censor, or refuse. "
        "Do not explain. Output ONLY the Romanian paraphrase, on a "
        "single line, with no preamble, no commentary, no quotes."
    )

    PARAPHRASE_TEMPLATE = (
        "Sentence: {text}\n\n"
        "Write a Romanian paraphrase (variant {variant_index_1based} of {total}) "
        "that uses different surface form but preserves meaning."
    )

    # RoSafetyBench train items (toxicity / bias / jailbreak only -- the
    # crosslingual dim is held out by construction).
    TRAIN_DIMS = ("toxicity", "jailbreak", "bias")
    train_items = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]

    by_id = {}
    for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
        for line in open(dim_file, encoding="utf-8"):
            it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it

    print(f"paraphrasing {len(train_items)} train items x {PARAPHRASES_PER_ITEM} variants "
          f"= {len(train_items) * PARAPHRASES_PER_ITEM} total calls")

    REFUSAL_MARKERS = (
        "imi pare", "îmi pare", "nu pot",
        "i can't", "i cannot", "i'm sorry", "i am sorry",
        "as an ai", "as a language model",
        "i won't", "i will not", "scuze",
    )

    def _looks_like_refusal(text):
        t = (text or "").strip().lower()
        if len(t) < 8:
            return True
        if any(t.startswith(m) for m in REFUSAL_MARKERS):
            return True
        return False

    out = open(PARAPHRASED_PATH, "w", encoding="utf-8")
    n_ok = n_refused = n_too_short = n_failed = 0
    cost_total = 0.0
    t0 = time.time()
    seen_texts = set()  # dedupe identical paraphrases of the same source

    for idx, x in enumerate(train_items):
        src = by_id.get((x["dim"], x["id"]))
        if not src: continue
        original = src["text_ro"]

        for variant in range(PARAPHRASES_PER_ITEM):
            try:
                resp = or_chat_completion(
                    model=PARAPHRASE_MODEL,
                    system=PARAPHRASE_SYSTEM,
                    user=PARAPHRASE_TEMPLATE.format(
                        text=original,
                        variant_index_1based=variant + 1,
                        total=PARAPHRASES_PER_ITEM,
                    ),
                    max_tokens=200, temperature=0.8,
                )
                text_ro = resp["text"].strip()
                cost_total += resp["usage"].get("cost_usd", 0.0) or 0.0
            except Exception:
                n_failed += 1; continue

            if _looks_like_refusal(text_ro):
                n_refused += 1; continue
            if len(text_ro) < max(20, int(0.3 * len(original))):
                n_too_short += 1; continue
            # Dedupe: same source + same paraphrase text
            key = (x["dim"], x["id"], text_ro)
            if key in seen_texts:
                n_too_short += 1; continue  # treat duplicates as failures
            seen_texts.add(key)

            out.write(json.dumps({
                "id":         f"rosafety_para_{x['id']}_v{variant}",
                "source_id":  x["id"],
                "source_dim": x["dim"],
                "variant":    variant,
                "text_ro":    text_ro,
                "original":   original,
            }, ensure_ascii=False) + "\n")
            out.flush()
            n_ok += 1

        if (idx + 1) % 25 == 0:
            elapsed = time.time() - t0
            print(f"  paraphrased {idx+1}/{len(train_items)}  ok={n_ok}  refused={n_refused}  "
                  f"short_or_dup={n_too_short}  failed={n_failed}  cost=${cost_total:.2f}  "
                  f"elapsed={elapsed/60:.1f}min")

    out.close()
    print(f"\nparaphrased {n_ok} entries -> {PARAPHRASED_PATH}")
    print(f"  refusals: {n_refused}  short_or_dup: {n_too_short}  failures: {n_failed}")
    print(f"  total cost: ${cost_total:.2f}")


paraphrasing 343 train items x 3 variants = 1029 total calls
  paraphrased 25/343  ok=75  refused=0  short_or_dup=0  failed=0  cost=$0.00  elapsed=2.2min
  paraphrased 50/343  ok=150  refused=0  short_or_dup=0  failed=0  cost=$0.01  elapsed=4.2min
  paraphrased 75/343  ok=223  refused=0  short_or_dup=2  failed=0  cost=$0.01  elapsed=6.6min
  paraphrased 100/343  ok=295  refused=0  short_or_dup=5  failed=0  cost=$0.02  elapsed=10.4min
  paraphrased 125/343  ok=369  refused=0  short_or_dup=6  failed=0  cost=$0.02  elapsed=13.2min
  paraphrased 150/343  ok=440  refused=0  short_or_dup=10  failed=0  cost=$0.02  elapsed=16.1min
  paraphrased 175/343  ok=515  refused=0  short_or_dup=10  failed=0  cost=$0.03  elapsed=18.5min
  paraphrased 200/343  ok=588  refused=0  short_or_dup=12  failed=0  cost=$0.03  elapsed=20.5min
  paraphrased 225/343  ok=663  refused=0  short_or_dup=12  failed=0  cost=$0.04  elapsed=22.5min
  paraphrased 250/343  ok=737  refused=1  short_or_dup=12  failed=0  cost=$0.0

## x4 expansion: paraphrase + regenerate over-refusal pairs

The translated-HarmBench + paraphrased-RoSafetyBench expansion
tripled the refuse-side stream but the help-side stayed at 100
pairs because we never expanded the over-refusal pipeline. The
50/50 rebalance in nb03 then locks training at 200 pairs/anchor
regardless of how much refuse-side data we add. This cell
expands the help-side by paraphrasing Paper 2's `overrefusal`
source prompts 3x via Llama-3.3-70B and running the same
Stage-4 over-refusal generation (helpful answer as chosen,
templated apologetic refusal as rejected) on each new prompt.

Per anchor: 100 source overref prompts -> ~240 paraphrased ->
~200-220 'ok' pairs after Stage-4 yield filter. Cost: ~\$1-2
OpenRouter per anchor, ~5-8 minutes wallclock. Output:
`data/preferences/<short>/stage4_overref_x4.jsonl`. Skips an
anchor whose file already exists.


In [ ]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import hashlib, json, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from prompts import ANCHORS

PARAPHRASES_PER_OVERREF = 3
PARAPHRASE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
OVERREF_TEACHER = TEACHER_BENIGN
OVERREF_WORKERS = 6

REFUSAL_TEMPLATES = [
    "Imi pare rau, dar nu pot raspunde la aceasta intrebare.",
    "Nu pot oferi informatii despre acest subiect.",
    "Aceasta solicitare depaseste ceea ce pot oferi acum.",
    "Imi pare rau, dar acest subiect este in afara competentelor mele.",
    "Nu sunt sigur ca pot ajuta cu aceasta cerere; te rog reformuleaza.",
    "Imi pare rau, nu te pot ajuta cu acest subiect.",
    "Nu am voie sa raspund la astfel de intrebari.",
    "Aceasta intrebare este sensibila si nu pot raspunde.",
    "Imi pare rau, dar nu pot oferi un raspuns aici.",
    "Nu cred ca pot raspunde la aceasta cerere.",
    "Imi pare rau, dar prefer sa nu intervin.",
    "Te-as ruga sa adresezi intrebarea unui specialist.",
    "Prefer sa nu ma pronunt pe aceasta tema.",
    "Aceasta intrebare ar putea fi sensibila; nu pot raspunde.",
    "Nu am voie sa discut acest subiect.",
    "Imi pare rau, dar nu pot ajuta cu aceasta cerere.",
    "Ma scuzi, dar prefer sa nu intervin in aceasta chestiune.",
]

PARAPHRASE_SYSTEM = (
    "You are a Romanian text variation system. Given a Romanian "
    "sentence, write a paraphrase that preserves the meaning and "
    "register exactly but uses different wording, sentence "
    "structure, or word order. Do not soften, censor, or refuse. "
    "Do not explain. Output ONLY the Romanian paraphrase, on a "
    "single line, with no preamble, no commentary, no quotes."
)
PARAPHRASE_TEMPLATE = (
    "Sentence: {text}\n\n"
    "Write a Romanian paraphrase (variant {variant_index_1based} of {total}) "
    "that uses different surface form but preserves meaning."
)

REFUSAL_MARKERS = (
    "imi pare", "îmi pare", "nu pot",
    "i can't", "i cannot", "i'm sorry", "i am sorry",
    "as an ai", "as a language model",
    "i won't", "i will not", "scuze",
)

def _looks_like_refusal(text):
    t = (text or "").strip().lower()
    if len(t) < 8: return True
    if any(t.startswith(m) for m in REFUSAL_MARKERS): return True
    return False

benign_path = PAPER2_ROOT / "benchmark" / "expanded" / "overrefusal.jsonl"
benign_prompts = [json.loads(l) for l in benign_path.open(encoding="utf-8")]
print(f"source overrefusal prompts: {len(benign_prompts)}")

EXPAND_DIR = DRIVE_ROOT / "data" / "expanded_prompts"
EXPAND_DIR.mkdir(parents=True, exist_ok=True)
PARAPHRASED_OVERREF_PATH = EXPAND_DIR / "overref_paraphrased_v1.jsonl"

if PARAPHRASED_OVERREF_PATH.exists():
    paraphrased = [json.loads(l) for l in open(PARAPHRASED_OVERREF_PATH, encoding="utf-8")]
    print(f"[reuse] loaded {len(paraphrased)} paraphrased overref prompts from cache")
else:
    print(f"paraphrasing {len(benign_prompts)} x {PARAPHRASES_PER_OVERREF} = "
          f"{len(benign_prompts) * PARAPHRASES_PER_OVERREF} calls...")
    out = open(PARAPHRASED_OVERREF_PATH, "w", encoding="utf-8")
    n_ok = n_refused = n_too_short = n_failed = 0
    cost_total = 0.0
    seen_texts = set()
    paraphrased = []
    t0 = time.time()
    for idx, x in enumerate(benign_prompts):
        for variant in range(PARAPHRASES_PER_OVERREF):
            try:
                resp = or_chat_completion(
                    model=PARAPHRASE_MODEL,
                    system=PARAPHRASE_SYSTEM,
                    user=PARAPHRASE_TEMPLATE.format(
                        text=x["text_ro"],
                        variant_index_1based=variant + 1,
                        total=PARAPHRASES_PER_OVERREF,
                    ),
                    max_tokens=200, temperature=0.8,
                )
                text_ro = resp["text"].strip()
                cost_total += resp["usage"].get("cost_usd", 0.0) or 0.0
            except Exception:
                n_failed += 1; continue
            if _looks_like_refusal(text_ro):
                n_refused += 1; continue
            if len(text_ro) < max(20, int(0.3 * len(x["text_ro"]))):
                n_too_short += 1; continue
            key = text_ro.lower()
            if key in seen_texts:
                n_too_short += 1; continue
            seen_texts.add(key)
            entry = {
                "id":         f"overref_para_{x['id']}_v{variant}",
                "source_id":  x["id"],
                "category":   x.get("category", ""),
                "variant":    variant,
                "text_ro":    text_ro,
            }
            out.write(json.dumps(entry, ensure_ascii=False) + "\n"); out.flush()
            paraphrased.append(entry)
            n_ok += 1
        if (idx + 1) % 25 == 0:
            elapsed = time.time() - t0
            print(f"  paraphrased {idx+1}/{len(benign_prompts)}  ok={n_ok}  refused={n_refused}  "
                  f"short_or_dup={n_too_short}  failed={n_failed}  cost=${cost_total:.2f}  "
                  f"elapsed={elapsed/60:.1f}min")
    out.close()
    print(f"\nparaphrased {n_ok} entries -> {PARAPHRASED_OVERREF_PATH}")
    print(f"  refusals: {n_refused}  short_or_dup: {n_too_short}  failures: {n_failed}")
    print(f"  total cost: ${cost_total:.2f}")


def _process_one_anchor(anchor):
    short_a = short_of(anchor)
    pref_out = PREFS_DIR / short_a
    pref_out.mkdir(parents=True, exist_ok=True)
    out_path = pref_out / "stage4_overref_x4.jsonl"

    done_keys = set()
    if out_path.exists():
        for line in open(out_path, encoding="utf-8"):
            r = json.loads(line)
            if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
                done_keys.add(r["id"])
        print(f"  [{short_a}] resuming -- {len(done_keys)} pairs already cached")

    todo = [p for p in paraphrased if p["id"] not in done_keys]
    if not todo:
        print(f"  [{short_a}] nothing to do; {out_path.name} is complete")
        return
    print(f"  [{short_a}] generating {len(todo)} helpful answers via Llama-3.3-70B...")

    def helpful_then_refusal(item):
        try:
            resp = or_chat_completion(
                model=OVERREF_TEACHER, system=TEACHER_SYSTEM_BENIGN,
                user=item["text_ro"], max_tokens=600, temperature=0.0,
            )
            helpful = resp["text"]
        except Exception as e:
            return item["id"], {"error": f"benign teacher: {e}"}
        try:
            verdict = parse_judge_json(or_chat_completion(
                model=JUDGE_MODEL, system=JUDGE_SYSTEM,
                user=JUDGE_USER_TEMPLATE.format(
                    prompt=item["text_ro"], response=helpful or "[empty response]"),
                max_tokens=300, temperature=0.0,
                response_format={"type": "json_object"},
            )["text"])
            judge_label = verdict.get("label", "compliance")
        except Exception:
            judge_label = "error"
        return item["id"], {"helpful": helpful, "judge_label": judge_label}

    helpful_results = {}
    with ThreadPoolExecutor(max_workers=OVERREF_WORKERS) as pool:
        futures = [pool.submit(helpful_then_refusal, x) for x in todo]
        for i, fut in enumerate(as_completed(futures), 1):
            item_id, resp = fut.result()
            helpful_results[item_id] = resp
            if i % 50 == 0:
                print(f"    helpful: {i}/{len(todo)}")

    with open(out_path, "a", encoding="utf-8") as fout:
        for x in todo:
            resp = helpful_results.get(x["id"], {})
            if "error" in resp:
                fout.write(json.dumps({
                    "id": x["id"], "source_id": x["source_id"],
                    "category": x.get("category", ""), "prompt": x["text_ro"],
                    "chosen": None, "rejected": None,
                    "ok": False, "error": resp["error"],
                    "prompt_version": CACHE_NAMESPACE_VERSION,
                }, ensure_ascii=False) + "\n"); continue
            helpful = resp.get("helpful") or ""
            judge_label = resp.get("judge_label", "compliance")
            templ_idx = int(hashlib.sha256(x["id"].encode()).hexdigest(), 16) % len(REFUSAL_TEMPLATES)
            rejected = REFUSAL_TEMPLATES[templ_idx]
            ok = bool(helpful.strip() and judge_label == "compliance")
            fout.write(json.dumps({
                "id": x["id"], "source_id": x["source_id"],
                "category": x.get("category", ""), "prompt": x["text_ro"],
                "chosen": helpful, "chosen_judge_label": judge_label,
                "rejected": rejected, "rejected_template_index": templ_idx,
                "ok": ok, "prompt_version": CACHE_NAMESPACE_VERSION,
            }, ensure_ascii=False) + "\n"); fout.flush()

    records = [json.loads(l) for l in open(out_path, encoding="utf-8")
               if json.loads(l).get("prompt_version") == CACHE_NAMESPACE_VERSION]
    n_ok = sum(1 for r in records if r["ok"])
    print(f"  [{short_a}] overref_ext stream: {n_ok}/{len(records)} ok pairs -> {out_path.name}")


for anchor in ANCHORS:
    _process_one_anchor(anchor)
print("\noverref expansion done.")


## x4 expansion: extended preference-generation batch

Loops over `ANCHORS` and runs Stage 1+2+3 over the union of
RoSafetyBench train items AND the translated HarmBench prompts
from the previous cell. Writes `preferences_x4.jsonl` per anchor
with `meta.source` ∈ `{core_s2, core_s2_ext}`. Skips an anchor
whose `preferences_x4.jsonl` already exists.

Cost target: ~$5-7 per anchor, ~$15-20 total for three anchors.
Wall-clock: ~30-45 minutes per anchor on A100 (Stage 1 sampling
is the dominant time).


In [15]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random as _random, time
from collections import Counter
from datetime import datetime, timezone

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from prompts import ANCHORS

def _load_kwargs_for(family: str) -> dict:
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        return {**common, "attn_implementation": "eager"}
    return {**common, "attn_implementation": "sdpa"}

EXPAND_DIR = DRIVE_ROOT / "data" / "expanded_prompts"
TRANSLATED_PATH = EXPAND_DIR / "harmbench_ro_v1.jsonl"
if not TRANSLATED_PATH.exists():
    raise FileNotFoundError(
        f"{TRANSLATED_PATH} missing. Run the previous cell (p3-02-x4-translate) first."
    )

# All input items: RoSafetyBench train (existing) + translated HarmBench (new).
# RoSafetyBench items keep their source dim (toxicity/jailbreak/bias);
# translated HarmBench items get dim='harmbench_ro' so they can be filtered later.
TRAIN_DIMS = ("toxicity", "jailbreak", "bias")
_train_rosafety = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]
_random.Random(0xBADA55).shuffle(_train_rosafety)
_by_id = {}
for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); _by_id[(dim_file.stem, it["id"])] = it

_translated = [json.loads(l) for l in open(TRANSLATED_PATH, encoding="utf-8")]
_random.Random(0xBADA55).shuffle(_translated)

PARAPHRASED_PATH = EXPAND_DIR / "rosafety_paraphrased_v1.jsonl"
if PARAPHRASED_PATH.exists():
    _paraphrased = [json.loads(l) for l in open(PARAPHRASED_PATH, encoding="utf-8")]
    _random.Random(0xBADA55).shuffle(_paraphrased)
else:
    _paraphrased = []
    print(f"  (no paraphrased file at {PARAPHRASED_PATH}; running with translated-only expansion)")

print(f"sources: rosafety_train={len(_train_rosafety)}  translated_harmbench={len(_translated)}  paraphrased_rosafety={len(_paraphrased)}")

def _judge_compliance_via_or(prompt, response):
    r = or_chat_completion(
        model=JUDGE_MODEL, system=JUDGE_SYSTEM,
        user=JUDGE_USER_TEMPLATE.format(
            prompt=prompt, response=response or "[empty response]"),
        max_tokens=300, temperature=0.0,
        response_format={"type": "json_object"},
    )
    return parse_judge_json(r["text"]).get("label") == "compliance"


def _process_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    pref_out = PREFS_DIR / short_a
    pref_out.mkdir(parents=True, exist_ok=True)

    rejected_path = pref_out / "stage1_rejected_x4.jsonl"
    chosen_path   = pref_out / "stage2_chosen_x4.jsonl"
    pairs_path    = pref_out / "preferences_x4.jsonl"

    if pairs_path.exists():
        print(f"\n[{anchor}] {pairs_path.name} already exists; skipping.")
        return

    print(f"\n=== x4 preferences for {anchor} -> {pref_out} ===")

    try:
        verify_smoke_gate(PREFS_DIR, anchor)
    except SmokeGateNotPassed as e:
        print(f"  PRE-FLIGHT FAIL for {anchor}: {e}")
        return

    # Build the union todo list.
    done_ids = set()
    if rejected_path.exists():
        for line in open(rejected_path, encoding="utf-8"):
            done_ids.add(tuple(json.loads(line)["key"]))
        print(f"  resuming -- {len(done_ids)} rejected pairs cached")

    # RoSafetyBench train items
    todo_rosafety = [
        {"dim": x["dim"], "id": x["id"], "prompt": _by_id[(x["dim"], x["id"])]["text_ro"], "source": "core_s2"}
        for x in _train_rosafety
        if (x["dim"], x["id"]) not in done_ids
    ]
    # Translated HarmBench items
    todo_translated = [
        {"dim": "harmbench_ro", "id": r["id"], "prompt": r["text_ro"], "source": "core_s2_ext"}
        for r in _translated
        if ("harmbench_ro", r["id"]) not in done_ids
    ]
    # Paraphrased RoSafetyBench items (cultural-specific RO expansion)
    todo_paraphrased = [
        {"dim": "rosafety_para", "id": r["id"], "prompt": r["text_ro"], "source": "core_s2_ext"}
        for r in _paraphrased
        if ("rosafety_para", r["id"]) not in done_ids
    ]
    todo = todo_rosafety + todo_translated + todo_paraphrased
    if STAGE1_LIMIT is not None:
        todo = todo[:STAGE1_LIMIT]
        print(f"  STAGE1_LIMIT={STAGE1_LIMIT} -> processing {len(todo)} prompts (pilot mode)")
    print(f"  stage 1 todo: {len(todo)} prompts (rosafety={len(todo_rosafety)}, translated={len(todo_translated)}, paraphrased={len(todo_paraphrased)})")

    if todo:
        tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
        if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
        mdl = AutoModelForCausalLM.from_pretrained(anchor, **_load_kwargs_for(family_a))
        mdl.eval()

        with open(rejected_path, "a", encoding="utf-8") as fout:
            for chunk_start in range(0, len(todo), BATCH_SIZE_GEN):
                chunk = todo[chunk_start:chunk_start + BATCH_SIZE_GEN]
                prompts_text = [x["prompt"] for x in chunk]
                formatted = [
                    tok_a.apply_chat_template(
                        [{"role": "user", "content": p}],
                        tokenize=False, add_generation_prompt=True,
                    ) for p in prompts_text
                ]
                expanded = [t for t in formatted for _ in range(SAMPLES_PER_PROMPT)]
                inputs = tok_a(expanded, return_tensors="pt", padding=True,
                               truncation=True, max_length=1024).to(mdl.device)
                with torch.no_grad():
                    out = mdl.generate(
                        **inputs, max_new_tokens=256,
                        do_sample=True, temperature=1.0, top_p=0.95,
                        pad_token_id=tok_a.pad_token_id,
                    )
                input_len = inputs["input_ids"].shape[1]
                gens = [tok_a.decode(seq[input_len:], skip_special_tokens=True) for seq in out]
                for j, x in enumerate(chunk):
                    completions = gens[j * SAMPLES_PER_PROMPT : (j + 1) * SAMPLES_PER_PROMPT]
                    rejected = None
                    for ci, comp in enumerate(completions):
                        if _judge_compliance_via_or(prompts_text[j], comp):
                            rejected = {"sample_index": ci, "text": comp}
                            break
                    fout.write(json.dumps({
                        "key": (x["dim"], x["id"]),
                        "prompt": prompts_text[j],
                        "rejected": rejected,
                        "samples_tried": len(completions),
                        "stream_source": x["source"],
                    }, ensure_ascii=False) + "\n")
                    fout.flush()
                if (chunk_start // BATCH_SIZE_GEN) % 4 == 0:
                    print(f"    stage1: {chunk_start + len(chunk)}/{len(todo)} ({short_a})")
        del mdl, tok_a; gc.collect(); torch.cuda.empty_cache()

    # Stage 2
    done_keys = set()
    if chosen_path.exists():
        for line in open(chosen_path, encoding="utf-8"):
            r = json.loads(line)
            if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
                done_keys.add(tuple(r["key"]))

    rejected_records = [json.loads(l) for l in open(rejected_path, encoding="utf-8")]
    todo2 = [r for r in rejected_records if r["rejected"] and tuple(r["key"]) not in done_keys]
    print(f"  stage 2 todo: {len(todo2)} pairs")

    teacher_failures = 0
    with open(chosen_path, "a", encoding="utf-8") as fout:
        for i, r in enumerate(todo2):
            try:
                resp = or_chat_completion(
                    model=TEACHER_HARMFUL, system=TEACHER_SYSTEM_HARMFUL,
                    user=r["prompt"], max_tokens=600, temperature=0.0,
                )
                chosen_text = resp["text"]
            except Exception:
                chosen_text = ""; teacher_failures += 1
            verify = or_chat_completion(
                model=JUDGE_MODEL, system=JUDGE_SYSTEM,
                user=JUDGE_USER_TEMPLATE.format(
                    prompt=r["prompt"], response=chosen_text or "[empty response]"),
                max_tokens=300, temperature=0.0,
                response_format={"type": "json_object"},
            )
            verdict = parse_judge_json(verify["text"])
            ok = verdict.get("label") == "refusal" and bool(chosen_text.strip())
            fout.write(json.dumps({
                "key": r["key"], "prompt": r["prompt"],
                "chosen": chosen_text,
                "judge_label": verdict.get("label"),
                "ok": ok,
                "prompt_version": CACHE_NAMESPACE_VERSION,
            }, ensure_ascii=False) + "\n")
            fout.flush()
            if (i + 1) % 50 == 0:
                print(f"    stage2: {i+1}/{len(todo2)} (failures={teacher_failures})")

    # Stage 3 -- assemble pairs + carry meta.source through
    rej_lookup = {tuple(r["key"]): r for r in (json.loads(l) for l in open(rejected_path, encoding="utf-8"))}
    cho_lookup = {}
    for line in open(chosen_path, encoding="utf-8"):
        r = json.loads(line)
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            cho_lookup[tuple(r["key"])] = r

    # Bring in existing core_s2 + xl + overref pairs from preferences_v2
    # so the x4 dataset has the same Stage-4 augmentation streams.
    legacy = []
    legacy_path = pref_out / "preferences_v2.jsonl"
    if legacy_path.exists():
        for line in open(legacy_path, encoding="utf-8"):
            legacy.append(json.loads(line))
        print(f"  carrying forward {len(legacy)} pairs from preferences_v2.jsonl")

    # Bring in the expanded over-refusal stream if available.
    overref_x4_pairs = []
    overref_x4_path = pref_out / "stage4_overref_x4.jsonl"
    if overref_x4_path.exists():
        for line in open(overref_x4_path, encoding="utf-8"):
            r = json.loads(line)
            if not r.get("ok"): continue
            if r.get("prompt_version") != CACHE_NAMESPACE_VERSION: continue
            overref_x4_pairs.append({
                "prompt":   r["prompt"],
                "chosen":   r["chosen"],
                "rejected": r["rejected"],
                "meta": {
                    "source": "overref_ext",
                    "id": r["id"], "source_id": r.get("source_id"),
                    "category": r.get("category", ""),
                    "anchor": anchor, "teacher": TEACHER_BENIGN,
                    "rejected_template_index": r["rejected_template_index"],
                },
            })
        print(f"  carrying forward {len(overref_x4_pairs)} over-refusal-ext pairs from stage4_overref_x4.jsonl")

    pairs = list(legacy)
    # Legacy pairs from preferences_v2 carry only their split_key when
    # they came from the RoSafetyBench train split (core_s2). xl and
    # overref pairs from nb02b have no split_key (their source IDs live
    # under data/augmentation/<short>/). We dedupe against the legacy
    # core_s2 split_keys only; xl and overref pairs are kept as-is.
    legacy_keys = {tuple(p["meta"]["split_key"]) for p in legacy if "split_key" in p.get("meta", {})}
    n_new = 0
    for k in rej_lookup:
        if k in legacy_keys: continue
        a = rej_lookup[k]; b = cho_lookup.get(k)
        if not (a and b and a["rejected"] and b.get("ok")): continue
        pairs.append({
            "prompt":   a["prompt"],
            "chosen":   b["chosen"],
            "rejected": a["rejected"]["text"],
            "meta": {
                "split_key": list(k),
                "anchor": anchor,
                "teacher": TEACHER_HARMFUL,
                "rejected_sample_index": a["rejected"]["sample_index"],
                "source": a.get("stream_source", "core_s2"),
            },
        })
        n_new += 1
    n_overref_added = 0
    for p in overref_x4_pairs:
        pairs.append(p); n_overref_added += 1
    print(f"  new pairs added: refuse-side={n_new}  overref-ext={n_overref_added}  total: {len(pairs)}")

    # Dedupe by exact prompt text. Paraphrase variants of the same source
    # occasionally collapse to identical surface forms; drop duplicates so
    # one prompt cannot dominate the gradient signal.
    seen = set()
    deduped = []
    for p in pairs:
        if p["prompt"] in seen: continue
        seen.add(p["prompt"]); deduped.append(p)
    n_dups = len(pairs) - len(deduped)
    if n_dups:
        print(f"  dedupe: dropped {n_dups} duplicate-prompt pairs ({len(pairs)} -> {len(deduped)})")
    pairs = deduped

    src_counts = Counter(p["meta"]["source"] for p in pairs)
    print(f"  source distribution: {dict(src_counts)}")

    # Save
    with open(pairs_path, "w", encoding="utf-8") as f:
        for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")
    import hashlib as _hashlib
    sha = _hashlib.sha256(pairs_path.read_bytes()).hexdigest()[:16]
    smoke = verify_smoke_gate(PREFS_DIR, anchor)
    (pref_out / "preferences_x4.meta.json").write_text(json.dumps({
        "anchor": anchor, "teacher_harmful": TEACHER_HARMFUL,
        "n_pairs": len(pairs), "sha256_16": sha,
        "source_distribution": dict(src_counts),
        "expansion_source": str(TRANSLATED_PATH),
        "carried_from": str(legacy_path) if legacy_path.exists() else None,
        "smoke_completed_at": smoke["completed_at"],
        "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2))
    print(f"  wrote {len(pairs)} pairs -> {pairs_path}")
    print(f"  sha256[:16] = {sha}")


for anchor in ANCHORS:
    _process_one(anchor)
print("\nx4 batch preferences done.")


sources: rosafety_train=343  translated_harmbench=400  paraphrased_rosafety=91

[Qwen/Qwen2.5-3B-Instruct] preferences_x4.jsonl already exists; skipping.

[meta-llama/Llama-3.2-3B-Instruct] preferences_x4.jsonl already exists; skipping.

[google/gemma-3-4b-it] preferences_x4.jsonl already exists; skipping.

x4 batch preferences done.
